# Week 8: NLP Foundations & Text Embeddings
## Notebook 12: Cross-Lingual Word Embedding Alignment (Orthogonal Procrustes)

### 1. Conceptual Intuition & Theoretical Background
Independently trained word embeddings across different languages (e.g. English and Spanish) exhibit similar geometric cluster structures, but their coordinate axes are rotated randomly relative to one another.

**The Orthogonal Procrustes Problem (Miklolov et al., 2013):**
Finds an orthogonal rotation matrix $W$ that aligns source language embeddings $X$ with target language embeddings $Y$ given a small bilingual dictionary of anchor words:
$$\min_{W} \|X W - Y\|_F^2 \quad \text{subject to } W^T W = I$$

**Analytical Solution via SVD:**
1. Compute the cross-covariance matrix:
$$M = Y^T X$$
2. Compute the SVD of $M$:
$$M = U \Sigma V^T$$
3. The optimal orthogonal transformation matrix is:
$$W = V U^T$$

Enables zero-shot cross-lingual transfer without retraining word embedding vectors.

---

### 2. Scratch Python Implementation: Orthogonal Procrustes Solver

In [ ]:
import numpy as np

def orthogonal_procrustes_scratch(X, Y):
    """
    Solves min ||X @ W - Y||_F subject to W^T W = I.
    X: Source embeddings (N, d)
    Y: Target embeddings (N, d)
    """
    # Cross-covariance matrix M = Y^T @ X
    M = np.dot(Y.T, X)
    
    # SVD of M
    U, _, Vt = np.linalg.svd(M)
    
    # Optimal orthogonal rotation W = V @ U^T
    W = np.dot(Vt.T, U.T)
    return W

# Synthetic Simulation: Align 2D English and Spanish word spaces
np.random.seed(42)
d = 4
n_anchor_words = 100

# True conceptual semantic representations
Z = np.random.normal(0, 1.0, (n_anchor_words, d))

# English space: Z + small distortion
X_english = Z + np.random.normal(0, 0.05, (n_anchor_words, d))

# Spanish space: True rotation matrix R_true applied to Z
Q, _ = np.linalg.qr(np.random.randn(d, d)) # Generate random orthogonal matrix
Y_spanish = np.dot(Z, Q) + np.random.normal(0, 0.05, (n_anchor_words, d))

# Distance before alignment
dist_before = np.linalg.norm(X_english - Y_spanish, 'fro')

# Solve Orthogonal Procrustes
W_align = orthogonal_procrustes_scratch(X_english, Y_spanish)
X_aligned = np.dot(X_english, W_align)

dist_after = np.linalg.norm(X_aligned - Y_spanish, 'fro')

# Check orthogonality constraint: W^T W == I
orthogonality_check = np.linalg.norm(np.dot(W_align.T, W_align) - np.eye(d))

print("--- Orthogonal Procrustes Cross-Lingual Alignment ---")
print(f"Frobenius Disparity Before Alignment: {dist_before:.2f}")
print(f"Frobenius Disparity After Alignment:  {dist_after:.2f}")
print(f"Alignment Error Reduction:            {(1 - dist_after / dist_before) * 100:.1f}%")
print(f"Orthogonality Matrix Residual:        {orthogonality_check:.1e} (W^T W == I)")

### 3. Step-by-Step Code Walkthrough
1. **Cross-Covariance ($M = Y^T X$):** Measures correlation across dimensions between paired anchor words.
2. **SVD Inversion:** Evaluates $W = V U^T$ to find the optimal rigid rotation.
3. **Orthogonality Preservation:** Preserves internal distances and angles in the original vector space: $\|u W - v W\| = \|u - v\|$.

---

### 4. Industry Applications & Trade-offs
- **Unsupervised Machine Translation (MUSE):** Mapping multilingual vector spaces without full sentence-level parallel corpora.
- **Trade-off:** Assumes exact isometric geometric alignment; fails between linguistically distant language families (e.g. English vs Chinese).

---

### 5. Top Interview Questions & Answers
**Q1: Why must the projection matrix $W$ in cross-lingual alignment be strictly orthogonal ($W^T W = I$)?**
> **Answer:** Orthogonal transformations are isometric: they preserve Euclidean distances and cosine similarities between words within the same language. Non-orthogonal transformations would distort monolingual semantic geometries.

**Q2: What is the Hubness Problem in high-dimensional nearest-neighbor retrieval?**
> **Answer:** In high dimensions, certain points ("hubs") appear as nearest neighbors to many unrelated queries. In cross-lingual retrieval, this is mitigated using **Cross-Domain Similarity Local Scaling (CSLS)**.